# Pronunciation Coach: fine-tune the phoneme recognizer (v2-3)

Fine-tunes `facebook/wav2vec2-lv-60-espeak-cv-ft` so that it outputs the sounds a speaker **actually said**,
including pronunciation errors, instead of the sounds they should have said.

- **Data:** the private Kaggle dataset made from `pronunciation-coach-finetune.zip` (see `scripts/package_kaggle.py`):
  L2-ARCTIC train/dev speakers (what the annotators heard) and CMU ARCTIC native US speakers.
- **Settings:** Accelerator **GPU T4** (one GPU is used; with "T4 x2" the second one is hidden), **Internet on**.
- **Time:** at most about 3 hours; training stops early when the dev F1 stops improving.
- **Output:** `/kaggle/working/recognizer-l2arctic.zip`: the best checkpoint (`best/`), an early one (`epoch3/`,
  only if a later epoch won) and the training log.

License: L2-ARCTIC is CC BY-NC 4.0, so the fine-tuned model is for non-commercial use only.
The base model is Apache-2.0. No token is needed; if one ever is, add it under *Add-ons → Secrets* as `HF_TOKEN`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # exactly one GPU, as in the smoke test ("T4 x2" has two)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4, then run again (on a CPU this would take days).")
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0), "| visible GPUs:", torch.cuda.device_count())

In [ ]:
import zipfile
from pathlib import Path

# Kaggle normally unpacks an uploaded zip; find the folder that holds manifests/.
# If the dataset still contains the zip itself, unpack it into /kaggle/working first.
found = list(Path("/kaggle/input").rglob("manifests/l2arctic_train.jsonl"))
if not found:
    archive = next(Path("/kaggle/input").rglob("pronunciation-coach-finetune.zip"))
    zipfile.ZipFile(archive).extractall("/kaggle/working/data")
    found = list(Path("/kaggle/working/data").rglob("manifests/l2arctic_train.jsonl"))
DATA = found[0].parent.parent
OUT = Path("/kaggle/working/recognizer")
print(DATA)
print(open(DATA / "README.txt").read())

## Packages
The versions the code was tested with (torch stays Kaggle's own). Training runs in a separate process,
so it uses these versions; the script also writes them to the training log.

In [ ]:
!pip install -q -r {DATA}/requirements-finetune.txt
!python -c "import transformers, tokenizers, numpy, scipy, soundfile; print('transformers', transformers.__version__, '| tokenizers', tokenizers.__version__, '| numpy', numpy.__version__, '| scipy', scipy.__version__, '| soundfile', soundfile.__version__)"

## Training
One epoch = twice the non-native sentences, sampled so that non-native and native speech weigh the same.
After every epoch the script prints the dev metrics (PER, MDD F1, native PER) and keeps the best checkpoint by F1.

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 CUDA_VISIBLE_DEVICES=0 python {DATA}/scripts/finetune_recognizer.py --data {DATA} --out {OUT}

## Download
Made before the plots, so a plotting problem cannot cost the model.

In [ ]:
import shutil

checkpoints = [p.name for p in OUT.iterdir() if p.is_dir()]
print("checkpoints:", checkpoints)
if checkpoints:
    archive = shutil.make_archive("/kaggle/working/recognizer-l2arctic", "zip", OUT)
    print(archive, round(Path(archive).stat().st_size / 1e6), "MB")
else:
    print("No checkpoint was saved: see the log above.")

## Curves

In [ ]:
import json

try:
    import matplotlib.pyplot as plt
    log = [json.loads(line) for line in open(OUT / "train_log.jsonl")]
    epochs = [e for e in log if "epoch" in e]
    for e in epochs:
        print(e)
    steps, losses = zip(*[(int(r.split(",")[0]), float(r.split(",")[2])) for r in open(OUT / "loss.csv").read().split()[1:]])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(steps, losses); ax[0].set_title("training loss (CTC)"); ax[0].set_xlabel("step")
    for key in ("l2_dev_f1", "l2_dev_per", "native_dev_per"):
        ax[1].plot([e["epoch"] for e in epochs], [e[key] for e in epochs], marker="o", label=key)
    ax[1].legend(); ax[1].set_xlabel("epoch")
    plt.savefig(OUT / "curves.png", dpi=100, bbox_inches="tight"); plt.show()
except Exception as error:   # the plots are nice to have; the model is already zipped
    print("Could not plot:", error)